# Fresh-smoke photochemistry from the published tables

This notebook replots selected main-text panels from *Ozone photochemistry in fresh biomass burning smoke over the United States* (Jin et al., 2026). The four compressed CSV files under `processed_data/` contain the archived observations and model output. Run all cells from the repository root or the `notebooks/` directory. Figures and summary tables appear inline.

The panel numbers follow the [paper](https://doi.org/10.1126/sciadv.ads2157). The four input tables are also archived on [Zenodo](https://doi.org/10.5281/zenodo.18226363).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display

# A notebook opened in Jupyter usually starts in the notebooks directory.
here = Path.cwd()
data_dir = next(
    (root / "processed_data" for root in (here, here.parent)
     if (root / "processed_data").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Run from the repository root or its notebooks directory.")

files = {
    "Observations": "all_data_obs.csv.gz",
    "MCM-BBVOC": "all_data_mcm_bbvoc.csv.gz",
    "MCM-GCVOC": "all_data_mcm_gcvoc.csv.gz",
    "GEOS-Chem": "all_data_gc.csv.gz",
}
tables = {}
for name, filename in files.items():
    path = data_dir / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing {filename} in {data_dir}. See the data DOI above.")
    tables[name] = pd.read_csv(path, low_memory=False).dropna(subset=["Flight_ID", "Plume_Age"])

obs = tables["Observations"]
styles = {"MCM-BBVOC": "-", "MCM-GCVOC": "--", "GEOS-Chem": ":"}
plumes = {
    "P-3B": ("Managed Understory", "#3299b5"),
    "RF03": ("Taylor Creek", "#800000"),
    "RF07": ("Donnell", "#d27800"),
    "RF09": ("Bear Trap", "#004cff"),
    "FN19": ("Black Water", "#686868"),
}
pan_flights = [flight for flight in plumes if flight != "P-3B"]
display(pd.Series({name: len(frame) for name, frame in tables.items()},
                  name="Rows with flight and physical age").to_frame())

### Plot conventions

Color identifies a plume, symbols show observations, and line style identifies a model. Physical age is the archived plume age in hours. Chemical age is the archived OH-equivalent age in hours. Binned curves below average values within each plume independently. An empty age interval produces a gap.

In [ ]:
flight_keys = [
    Line2D([], [], color=color, marker="o", linestyle="none", label=f"{flight} · {name}")
    for flight, (name, color) in plumes.items()
]
model_keys = [Line2D([], [], color="black", linestyle=style, label=name)
              for name, style in styles.items()]

def age_bins(frame, value, minutes, x="Plume_Age", midpoints=False):
    """Average paired data by physical age. Optionally plot right-closed bin centers."""
    data = frame[["Plume_Age", value]].copy()
    if x != "Plume_Age":
        data[x] = frame[x]
    data = data.replace([np.inf, -np.inf], np.nan).dropna()
    if data.empty:
        return [], []
    width = minutes / 60
    if not midpoints:
        bin_index = data.Plume_Age // width
        means = data.groupby(bin_index)[[x, value]].mean()
        means = means.reindex(range(int(bin_index.min()), int(bin_index.max()) + 1))
        return means[x].to_numpy(), means[value].to_numpy()
    first = int(np.floor(data.Plume_Age.min() / width))
    last = int(np.ceil(data.Plume_Age.max() / width))
    edges = np.arange(first, max(first + 1, last) + 1) * width
    data["bin"] = pd.cut(data.Plume_Age, edges, right=True)
    means = data.groupby("bin", observed=False)[[x, value]].mean()
    return (edges[:-1] + edges[1:]) / 2, means[value].to_numpy()

def plot_series(ax, value, *, flights=None, x="Plume_Age", models=None,
                minutes=None, midpoints=False, observations=None, measured=True, scale=1):
    flights = list(plumes) if flights is None else list(flights)
    models = list(styles) if models is None else list(models)
    measurements = obs if observations is None else observations
    for flight in flights:
        color = plumes[flight][1]
        for name in models:
            group = tables[name].loc[tables[name].Flight_ID == flight].sort_values("Plume_Age")
            if minutes is None:
                xx, yy = group[x], group[value]
            else:
                xx, yy = age_bins(group, value, minutes, x, midpoints)
            ax.plot(xx, np.asarray(yy) / scale, styles[name], color=color, linewidth=1.4)
        if measured:
            group = measurements.loc[measurements.Flight_ID == flight, [x, value]]
            group = group.replace([np.inf, -np.inf], np.nan).dropna()
            ax.scatter(group[x], group[value] / scale, color=color, s=23, zorder=3,
                       edgecolors="white", linewidths=0.35)
    ax.set_xlabel("Physical age (h)" if x == "Plume_Age" else "Chemical age (h)")
    ax.grid(axis="y", alpha=0.2)

def finish(fig, flights=None, models=None):
    flights = list(plumes) if flights is None else flights
    models = list(styles) if models is None else models
    keys = [key for flight, key in zip(plumes, flight_keys) if flight in flights]
    keys += [key for name, key in zip(styles, model_keys) if name in models]
    fig.legend(handles=keys, loc="lower center", ncol=min(4, len(keys)),
               frameon=False, fontsize=8)
    fig.tight_layout(rect=(0, 0.16, 1, 1))
    plt.show()

## Figure 1 · Hydroxyl radical and plume age

Panel A pairs OH inferred from VOC decay for three WE-CAN flights with 15-minute model means for all five plumes. Four OH points are excluded at the physical ages flagged in the original analysis. The horizontal line marks the assumed ambient OH concentration. Black symbols show literature values transcribed in the original figure code.

Panel B pools the five MCM-BBVOC flights, uses the 30-second physical-age bins in the original plotting code, and fits a line through the origin within each hour. Set `age_bin_minutes` to 1 to use the one-minute bins described in the caption.

In [ ]:
age_bin_minutes = 0.5
oh_obs = obs[obs.Flight_ID.isin(["RF03", "RF07", "RF09"])].copy()
excluded_ages = {"RF07": [3.541500], "RF09": [3.744000, 4.620500, 4.093667]}
for flight, ages in excluded_ages.items():
    flagged = (oh_obs.Flight_ID == flight) & oh_obs.Plume_Age.round(6).isin(ages)
    oh_obs.loc[flagged, "cal_OH_mean"] = np.nan

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.8))
plot_series(left, "output_OH", minutes=15, measured=False, scale=1e6)
plot_series(left, "cal_OH_mean", models=[], observations=oh_obs, scale=1e6)
literature = [
    ("Hobbs et al.", "*", [40 / 60], [17]),
    ("Yokelson et al.", "s", [40 / 60, 1.2], [11.4, 4.1]),
    ("Palm et al.", "P", [20 / 60], [4]),
    ("Akherati et al.", "^", [21 / 60, 45 / 60, 53 / 60, 56 / 60, 32 / 60],
     [8.9, 4, 3, 2.8, 3.5]),
]
for label, marker, ages, values in literature:
    left.scatter(ages, values, color="black", marker=marker, s=30, label=label, zorder=4)
left.axhline(1.5, color="black", linewidth=0.9)
left.set(xlim=(0, 5), ylim=(0, 25), ylabel="OH (10⁶ molecules cm⁻³)", title="A   Plume-center OH")
left.legend(fontsize=7, frameon=False)

model = tables["MCM-BBVOC"]
ages = model.loc[model.Flight_ID.isin(plumes), ["Plume_Age", "output_chem_age"]].dropna()
for hour, color in enumerate(["tab:red", "tab:green", "tab:blue", "tab:orange", "tab:gray"]):
    part = ages[(ages.Plume_Age >= hour) & (ages.Plume_Age < hour + 1)]
    points = part.groupby(part.Plume_Age // (age_bin_minutes / 60)).mean()
    if len(points) < 2:
        continue
    x, y = points.Plume_Age.to_numpy(), points.output_chem_age.to_numpy()
    slope = np.dot(x, y) / np.dot(x, x)
    right.scatter(x, y, color=color, s=7, alpha=0.55)
    right.plot([0, y.max() / slope], [0, y.max()], color=color,
               label=f"{hour}–{hour + 1} h, slope {slope:.1f}")
limit = np.ceil(ages.output_chem_age.max())
right.plot([0, limit], [0, limit], "k--", linewidth=0.8, label="1:1")
right.set(xlim=(0, limit), ylim=(0, limit), xlabel="Physical age (h)", ylabel="Chemical age (h)",
          title="B   Chemical versus physical age")
right.legend(fontsize=8, frameon=False)
finish(fig)

## Figure 2 · Radical sources and VOC reactivity

For each photolysis source, the calculation multiplies mean concentration by mean photolysis frequency using rows with both measurements in each half-hour bin, then applies the radical yield. The original plotting code pools the full observation table for panels A and B. This reproduces the first three published HONO fractions (82%, 76% and 48%) and the late-age bar. For that last bar the code averages the half-hour rates at ages of 5 h or more, then calculates source fractions.

The published caption describes five selected plumes. Set `budget_flights = list(plumes)` to examine that subset. Its oldest observation is 4.93 h. The sample-size table counts the concentration–photolysis pairs contributing to each source. A missing source measurement contributes no bar segment.

Panel C shows observed and modeled CO-normalized VOC OH reactivity for each plume.

In [ ]:
budget_flights = None
budget_obs = obs if budget_flights is None else obs[obs.Flight_ID.isin(budget_flights)]
sources = {
    "HONO": ("HONO", "JHNO2_OH_NO", 1),
    "Ozone": ("O3", "JO3_O2_O1D", 0.5),
    "Formaldehyde": ("CH2O", "JCH2O_H_HCO", 2),
    "Glyoxal": ("GLYX", "JCHOCHO_HCO_HCO", 2),
    "Diacetyl": ("BIACET", "J23Butanedione_Products", 0.6),
    "Acetaldehyde": ("ALD2", "JCH3CHO_CH3_HCO", 2),
    "Methylglyoxal": ("MGLY", "JCH3COCHO_CH3CO_HCO", 1.3),
    "Glycolaldehyde": ("GLYC", "JCH2O_H_HCO", 2 * 0.346),
}
rates, counts = {}, {}
for label, (species, frequency, radical_yield) in sources.items():
    pairs = budget_obs[["Plume_Age", species, frequency]].dropna()
    groups = pairs.groupby(np.floor(pairs.Plume_Age / 0.5).astype(int))
    means = groups[[species, frequency]].mean()
    rates[label] = 1000 * radical_yield * means[species] * means[frequency]
    counts[label] = groups.size()

rates, counts = pd.DataFrame(rates), pd.DataFrame(counts)
young_bins = list(range(10))
rates_young, counts_young = rates.reindex(young_bins), counts.reindex(young_bins)
bin_labels = [f"{i / 2:g}–{(i + 1) / 2:g}" for i in young_bins]
if (rates.index >= 10).any():
    rates_young.loc[10] = rates[rates.index >= 10].mean()
    counts_young.loc[10] = counts[counts.index >= 10].sum()
    bin_labels.append("≥5")
rates_young.index = counts_young.index = pd.Index(bin_labels, name="Physical age (h)")
relative = rates_young.div(rates_young.sum(axis=1, min_count=1), axis=0) * 100

fig, axes = plt.subplots(3, 1, figsize=(9.5, 9))
for ax, frame, label in [(axes[0], rates_young, "A   HOₓ production"),
                          (axes[1], relative, "B   Relative source contributions")]:
    frame.plot.bar(stacked=True, ax=ax, legend=False, rot=0, width=0.8)
    ax.set_title(label, loc="left")
    ax.grid(axis="y", alpha=0.15)
axes[0].set(xlabel="", ylabel="P(HOₓ) (ppt s⁻¹)")
axes[0].legend(ncol=4, fontsize=7, frameon=False)
axes[1].set(ylabel="Share (%)", ylim=(0, 100))
for bars in axes[1].containers:
    axes[1].bar_label(bars, labels=[f"{bar.get_height():.0f}" if bar.get_height() > 6 else ""
                                   for bar in bars], label_type="center", fontsize=7)
plot_series(axes[2], "VOCR: CO")
axes[2].set(xlim=(0, 5), ylabel="VOC OH reactivity (s⁻¹ ppm CO⁻¹)",
            title="C   CO-normalized VOC OH reactivity")
finish(fig)
display(counts_young.fillna(0).astype(int))

## Figure 3 · Ozone enhancement per unit CO

The left panel uses physical age and all three models. The right panel uses OH-equivalent chemical age and MCM-BBVOC, averaging the model in 15-minute bins of physical age. The ozone and PAN panels use the archived normalized excess mixing ratios (NEMRs), in ppbv per ppbv.

Normalized mean bias (NMB) compares each model with an observation fit evaluated at its model time steps, as in the original figure code. The fit is quadratic for all flights except FN19, where it is linear. NMB is 100 times the sum of model minus fitted values divided by the sum of fitted values. The table below each figure reports percentages calculated over the full archived time series for each flight.

In [ ]:
def excess_ratio_figure(variable, ylabel, flights):
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)
    plot_series(left, variable, flights=flights)
    plot_series(right, variable, flights=flights, x="output_chem_age",
                models=["MCM-BBVOC"], minutes=15)
    left.set(xlim=(0, 5), ylabel=ylabel, title="A   Physical age")
    right.set(xlim=(0, None), title="B   Chemical age")
    left.set_ylim(bottom=0)
    finish(fig, flights=flights)

def normalized_bias(observed, obs_column, model_column, flights, minutes=None):
    result = {}
    for flight in flights:
        points = observed.loc[observed.Flight_ID == flight, ["Plume_Age", obs_column]]
        points = points.replace([np.inf, -np.inf], np.nan).dropna()
        degree = 1 if flight == "FN19" else 2
        if points.Plume_Age.nunique() <= degree:
            continue
        fit = np.polynomial.Polynomial.fit(points.Plume_Age, points[obs_column], degree)
        result[flight] = {}
        for name in styles:
            group = tables[name].loc[tables[name].Flight_ID == flight, ["Plume_Age", model_column]]
            group = group.replace([np.inf, -np.inf], np.nan).dropna()
            x, y = (group.Plume_Age.to_numpy(), group[model_column].to_numpy())
            if minutes is not None:
                x, y = age_bins(group, model_column, minutes)
            good = np.isfinite(x) & np.isfinite(y)
            predicted = fit(np.asarray(x)[good])
            denominator = predicted.sum()
            if denominator != 0:
                result[flight][name] = 100 * (np.asarray(y)[good] - predicted).sum() / denominator
    return pd.DataFrame.from_dict(result, orient="index").rename_axis("NMB (%)").round(1)

excess_ratio_figure("NEMR_O3", "ΔO₃ / ΔCO (ppbv/ppbv)", list(plumes))
display(normalized_bias(obs, "NEMR_O3", "NEMR_O3", plumes))

## Figure 4 · Formaldehyde-to-NO₂ ratio

The age panel shows four plumes with measured formaldehyde-to-NO₂ ratios (FNRs). The NO panel uses finite NO and positive FNR pairs from the archived observation table, colored by FNR interval. Horizontal guides mark FNR = 1 and the 4–6 range.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(10.5, 4.6), sharey=True)
fnr = "CH2O: NO2"
fnr_flights = [flight for flight in plumes if flight != "P-3B"]
plot_series(left, fnr, flights=fnr_flights, minutes=10, midpoints=True)
left.set(xlim=(0, 5), ylabel="HCHO / NO₂ (ppbv/ppbv)", title="A   Plume evolution")

pairs = obs[["NO", fnr]].replace([np.inf, -np.inf], np.nan).dropna()
edges = [0, 2, 4, 6, 8, 10, np.inf]
intervals = pd.cut(pairs[fnr], edges, labels=["0–2", "2–4", "4–6", "6–8", "8–10", ">10"])
palette = plt.get_cmap("viridis")(np.linspace(0.12, 0.88, len(edges) - 1))
for label, color in zip(intervals.cat.categories, palette):
    group = pairs[intervals == label]
    right.scatter(group.NO, group[fnr], color=color, s=19, label=label)
right.set(xlabel="NO (ppbv)", title="B   Dependence on NO")
right.legend(title="HCHO / NO₂", fontsize=7, title_fontsize=8, ncol=2, frameon=False)
for ax in (left, right):
    ax.axhline(1, color="black", linestyle="--", linewidth=0.8)
    ax.axhspan(4, 6, color="gray", alpha=0.22)
finish(fig, flights=fnr_flights)

## Figure 5 · Indicators of ozone production sensitivity

Panel A is the ratio of NOₓ OH reactivity to VOC OH reactivity. Reference lines mark 0.04 and 0.2. Panel B compares radical loss by radical–radical versus radical–NOₓ reactions. MCM-BBVOC is the simulation named in the caption. The figure code also draws MCM-GCVOC and GEOS-Chem. The line styles match panel A, and the reference ratio is 1. Model points in B are first averaged over ten minutes and then smoothed into ten-minute intervals for P-3B or thirty-minute intervals for the other flights.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.4))
plot_series(left, "OHRnox: OHRvoc", minutes=10, midpoints=True)
left.set(xlim=(0, 5), ylim=(0, 0.22), ylabel="OH reactivity ratio",
         title="A   NOₓ to VOC OH reactivity")
for threshold in [0.04, 0.2]:
    left.axhline(threshold, color="black", linewidth=0.8)
loss = "LROx: LNOx"
for flight, (_, color) in plumes.items():
    for name, style in styles.items():
        group = tables[name].loc[tables[name].Flight_ID == flight]
        x, y = age_bins(group, loss, 10)
        means = pd.DataFrame({"Plume_Age": x, loss: y})
        x, y = age_bins(means, loss, 10 if flight == "P-3B" else 30, midpoints=True)
        right.plot(x, y, style, color=color, linewidth=1.4)
right.axhline(1, color="black", linewidth=0.8)
right.set(xlim=(0, 5), xlabel="Physical age (h)", ylabel="LROₓ / LNOₓ",
          title="B   Radical loss pathways")
right.grid(axis="y", alpha=0.2)
finish(fig)

## Figure 6 · PAN enhancement per unit CO

The original figure excludes the Managed Understory flight (P-3B) because a PAN measurement was unavailable. The archived table contains numerical zeros for its PAN ratio. The panels therefore show four flights. The final table distinguishes the five-flight correlation calculated from the archived numeric entries from the correlation among the four flights plotted here.

In [ ]:
excess_ratio_figure("NEMR_PAN", "ΔPAN / ΔCO (ppbv/ppbv)", pan_flights)
display(normalized_bias(obs, "NEMR_PAN", "NEMR_PAN", pan_flights))

### Chemical-age correlations and model biases

R² is the squared Pearson correlation between observed NEMR and archived chemical age, after dropping rows missing either value. Each row is one aircraft sample. The PAN rows below show how the treatment of P-3B affects the result.

In [ ]:
def chemical_age_correlation(variable, flights):
    pairs = obs.loc[obs.Flight_ID.isin(flights), ["output_chem_age", variable]]
    pairs = pairs.replace([np.inf, -np.inf], np.nan).dropna()
    return {"Paired samples": len(pairs),
            "R²": pairs["output_chem_age"].corr(pairs[variable]) ** 2}

correlations = pd.DataFrame.from_dict({
    "O₃ · five study flights": chemical_age_correlation("NEMR_O3", plumes),
    "PAN · five flights including archived P-3B zeros": chemical_age_correlation("NEMR_PAN", plumes),
    "PAN · four flights with measurements": chemical_age_correlation("NEMR_PAN", pan_flights),
}, orient="index")
display(correlations.astype({"Paired samples": int}).round(4))

print("Figure 1A · OH normalized mean bias")
display(normalized_bias(oh_obs, "cal_OH_mean", "output_OH", ["RF03", "RF07", "RF09"], minutes=15))
print("Figure 2C · VOC OH reactivity normalized mean bias")
display(normalized_bias(obs, "VOCR: CO", "VOCR: CO", plumes))